## Sentiment Analysis：
- Evaluate text for overall sentiment: positive, negative, neutral.
- Compare sentiment with the event or information being described.

In [5]:
import re
from time import time
import numpy as np
import pandas as pd
import csv
import warnings
from nrclex import NRCLex
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Scikit-learn imports
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

In [6]:
# Data loading and preprocessing functions
COLS = ["id","label","statement","subjects","speaker","job_title",
        "state_info","party_affiliation","barely_true_cnt","false_cnt",
        "half_true_cnt","mostly_true_cnt","pants_on_fire_cnt","context","justification"]

data_path = '../data/'

def read_tsv(path):
    """Load TSV data with proper handling of quotes and escape characters"""
    return pd.read_csv(path, sep="\t", header=None, names=COLS,
                       engine="python", quoting=csv.QUOTE_NONE, escapechar="\\",
                       on_bad_lines="skip")

def text_of(r):
    """Combine statement, context, and justification into single text"""
    return " ".join([str(r.get("statement","")), str(r.get("context","")), str(r.get("justification",""))]).strip()

def first_subject(s):
    """Extract first subject from subjects field"""
    parts = re.split(r"[;,]", s) if isinstance(s,str) else []
    return parts[0].strip().lower() if parts and parts[0].strip() else "unknown"

df_tr = read_tsv(data_path + 'train2.tsv')
df_te = read_tsv(data_path + 'test2.tsv')
df_va = read_tsv(data_path + 'val2.tsv')

print("Data loading functions defined!")

Data loading functions defined!


In [7]:
# Sentiment analysis constants
ALPHA = 1.0
EPS = 1e-8
EMOTIONS_TO_ANALYZE = ["positive", "negative"]
_token_re = re.compile(r"[A-Za-z']+")

def _word_count(text):
    return len(_token_re.findall(str(text)))

def nrc_doc_score_from_text(text, alpha: float = ALPHA):
    emo = NRCLex(str(text))
    pos = emo.raw_emotion_scores.get('positive', 0)
    neg = emo.raw_emotion_scores.get('negative', 0)
    return float(np.log((pos + alpha) / (neg + alpha)))

def get_sentiment_class(vader_row):
    c = float(vader_row.get("compound", 0.0))
    if c >= 0.05:
        return 'Positive'
    elif c <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

class ModelArtifacts:
    def __init__(self, vectorizer, topic_model, topic_mu, topic_sigma):
        self.vectorizer = vectorizer
        self.topic_model = topic_model
        self.topic_mu = topic_mu
        self.topic_sigma = topic_sigma

def train_sentiment_models(train_df, text_col="statement"):
    train_texts = train_df[text_col].fillna("").astype(str).tolist()

    vectorizer = CountVectorizer(max_df=0.9, min_df=5, stop_words='english')
    X_train = vectorizer.fit_transform(train_texts)

    topic_model = LatentDirichletAllocation(n_components=20, random_state=42, learning_method="batch")
    topic_model.fit(X_train)

    theta_train = topic_model.transform(X_train)
    s_train = np.array([nrc_doc_score_from_text(s) for s in train_texts])

    weights_sum = theta_train.sum(axis=0) + EPS
    topic_mu = (theta_train.T @ s_train) / weights_sum

    diffs = s_train[:, None] - topic_mu[None, :]
    topic_var = (theta_train * diffs ** 2).sum(axis=0) / weights_sum
    topic_sigma = np.sqrt(topic_var + EPS)

    return ModelArtifacts(
        vectorizer=vectorizer,
        topic_model=topic_model,
        topic_mu=topic_mu,
        topic_sigma=topic_sigma
    )

def extract_sentiment_features_from_statement(statement: str, artifacts, analyzer) -> dict:
    s = "" if statement is None else str(statement)
    wc = _word_count(s)

    emo_obj = NRCLex(s)
    emo_counts = {e: int(emo_obj.raw_emotion_scores.get(e, 0)) for e in EMOTIONS_TO_ANALYZE}
    emo_densities = {f"{e}_density": (emo_counts[e] / wc) if wc > 0 else 0.0 for e in EMOTIONS_TO_ANALYZE}

    emotion_logratio = float(np.log((emo_counts["positive"] + ALPHA) / (emo_counts["negative"] + ALPHA)))
    s_val = emotion_logratio

    X = artifacts.vectorizer.transform([s])
    theta = artifacts.topic_model.transform(X)

    mu_hat = float((theta @ artifacts.topic_mu)[0])
    var_hat = float((theta @ (artifacts.topic_sigma ** 2))[0] + EPS)
    sd_hat = float(np.sqrt(var_hat))

    sent_dev_diff = float(s_val - mu_hat)
    sent_dev_z = float(sent_dev_diff / sd_hat) if sd_hat > 0 else 0.0

    vader = analyzer.polarity_scores(s)
    sentiment_value = get_sentiment_class(vader)
    sentiment_extremity = abs(float(vader.get("compound", 0.0)))

    return {
        "word_count": wc,
        "emotion_logratio": float(emotion_logratio),
        "sent_dev_diff": float(sent_dev_diff),
        "sent_dev_z": float(sent_dev_z),
        "vader_pos": float(vader.get("pos", 0.0)),
        "vader_neu": float(vader.get("neu", 0.0)),
        "vader_neg": float(vader.get("neg", 0.0)),
        "vader_compound": float(vader.get("compound", 0.0)),
        "sentiment_value": sentiment_value,
        "sentiment_extremity": float(sentiment_extremity),
    }

# Train sentiment model
sentiment_artifacts = train_sentiment_models(df_tr)
vader_analyzer = SentimentIntensityAnalyzer()

def create_sentiment_feature_dataframe(df: pd.DataFrame,
                             artifacts: ModelArtifacts,
                             analyzer: SentimentIntensityAnalyzer,
                             text_col="statement"):

    feature_rows = df[text_col].apply(
        extract_sentiment_features_from_statement,
        args=(artifacts, analyzer)
    )

    features_df = pd.DataFrame(feature_rows.tolist())
    return features_df

train_sentiment_features = create_sentiment_feature_dataframe(df_tr, sentiment_artifacts, vader_analyzer)
test_sentiment_features = create_sentiment_feature_dataframe(df_te, sentiment_artifacts, vader_analyzer)
validation_sentiment_features = create_sentiment_feature_dataframe(df_va, sentiment_artifacts, vader_analyzer)

features = ["emotion_logratio", "sent_dev_z"]

X_tv = pd.concat([train_sentiment_features[features], validation_sentiment_features[features]], axis=0)
y_tv = pd.concat([train_sentiment_features["sentiment_value"], validation_sentiment_features["sentiment_value"]], axis=0)
X_te = test_sentiment_features[features]
y_te = test_sentiment_features["sentiment_value"]

In [8]:
names = [
    "Nearest Neighbors", "Linear SVM", "RBF SVM", 
    "Decision Tree", "Random Forest", "Neural Net", "AdaBoost",
    "Naive Bayes", "QDA"
]

classifiers = [
    KNeighborsClassifier(3),
    SVC(kernel="linear", C=0.025, random_state=42),
    SVC(gamma=2, C=1, random_state=42),
    DecisionTreeClassifier(max_depth=5, random_state=42),
    RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1, random_state=42),
    MLPClassifier(alpha=1, max_iter=1000, random_state=42),
    AdaBoostClassifier(algorithm='SAMME', random_state=42),
    GaussianNB(),
    QuadraticDiscriminantAnalysis()
]

results = []
best_idx = None
best_cv_f1 = -1.0
best_pipe = None

print(f"{'Classifier':<20} {'CV Acc':<10} {'CV F1':<10} {'Time (s)':<10}")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'f1_macro']

for idx, (name, clf) in enumerate(zip(names, classifiers)):
    start_time = time()
    pipe = make_pipeline(StandardScaler(), clf)

    cv_results = cross_validate(pipe, X_tv, y_tv, cv=cv, scoring=scoring_metrics, n_jobs=-1)

    cv_acc = 100.0 * cv_results['test_accuracy'].mean()
    cv_f1  = cv_results['test_f1_macro'].mean()
    training_time = time() - start_time

    print(f"{name:<20} {cv_acc:>8.2f}% {cv_f1:>9.4f} {training_time:>8.2f}")

    results.append({
        "name": name,
        "cv_acc": cv_acc,
        "cv_f1": cv_f1,
        "time": training_time,
    })

    if cv_f1 > best_cv_f1:
        best_cv_f1 = cv_f1
        best_idx = idx
        best_pipe = pipe

print("=" * 60)
print(f"Best by CV F1: {names[best_idx]} (CV F1 = {best_cv_f1:.4f})")

# Refit best model on full Train+Val and evaluate once on Test
best_pipe.fit(X_tv, y_tv)
y_pred_te = best_pipe.predict(X_te)

test_acc = 100.0 * accuracy_score(y_te, y_pred_te)
test_f1  = f1_score(y_te, y_pred_te, average='macro')

print(f"Test Accuracy of best model: {test_acc:.2f}%")
print(f"Test Macro-F1 of best model: {test_f1:.4f}")

Classifier           CV Acc     CV F1      Time (s)  
Nearest Neighbors       41.52%    0.4105     1.51
Linear SVM              49.37%    0.4951     2.09
RBF SVM                 49.84%    0.4997     1.79
Decision Tree           49.72%    0.4950     0.08
Random Forest           49.95%    0.5000     0.11
Neural Net              49.18%    0.4929     0.55
AdaBoost                49.65%    0.4979     0.32
Naive Bayes             48.65%    0.4868     0.03
QDA                     48.97%    0.4893     0.03
Best by CV F1: Random Forest (CV F1 = 0.5000)
Test Accuracy of best model: 51.19%
Test Macro-F1 of best model: 0.5080
